# E2 · Notebook 1 — Entorno, datos Sen1Floods11, línea base y teardown

**NO EJECUTADO.** Solo E2. Parte 1 de 2: deja en **tu Drive** (`MyDrive/auto-prompt-sam2`) los datos y la evidencia de la línea base (SAM 2.1 tiny congelado) y del teardown. Reutiliza el código del repositorio (`prepare_stage2.py`, `run_stage3.py`, `run_stage4.py`).

* Runtime: **T4 GPU**. Todo el código se clona desde GitHub (rama `_E.V_`) y su commit queda fijado en tu Drive.
* Los datos, pesos, cachés y checkpoints se guardan solo en tu Drive, nunca en GitHub.
* Referencia asistida por etiquetas (no desplegable). No se usa el flujo automático DINOv3→SAM.
* La evidencia ejecutada de tu compañero en `docs/project/evidence/` es antecedente; **no acredita tu ejecución**.
* Licencia de Sen1Floods11: respuesta del publicador pendiente.

## 1. Entorno
Clona el código, fija el commit, crea el entorno aislado y lee `HF_TOKEN` de los secretos de Colab (sin mostrarlo).

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
GITHUB_URL = 'https://github.com/roberto-chavez/auto-prompt-sam2.git'
BRANCH = '_E.V_'
PINNED_COMMIT = ''    # '' = usa el commit ya fijado en tu Drive; en la 1.ª ejecución fija la punta actual de la rama
UPDATE_CODE = False   # True SOLO para adoptar código nuevo de la rama (entonces rehaz E2 desde 'prepare')
PERSIST_ROOT = '/content/drive/MyDrive/auto-prompt-sam2'   # datos, pesos, cachés y checkpoints: SOLO aquí
REPO = Path('/content/auto-prompt-sam2')
PYTHON = Path('/content/auto-prompt-venv/bin/python')
Path(PERSIST_ROOT).mkdir(parents=True, exist_ok=True)

def sh(args, cwd=None):
    r = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    if r.returncode:
        raise RuntimeError(f'{args[:3]} falló:\n{r.stdout}\n{r.stderr}')
    return r.stdout.strip()

# --- Código desde GitHub, en una revisión fijada y reproducible --------------------------------------
PIN_FILE = Path(PERSIST_ROOT) / 'e2/code_commit.txt'
existed = REPO.exists()
if not existed:
    sh(['git', 'clone', '--branch', BRANCH, GITHUB_URL, str(REPO)])
elif UPDATE_CODE:
    sh(['git', 'fetch', 'origin', BRANCH], cwd=REPO)
want = PINNED_COMMIT.strip() or ('' if UPDATE_CODE or not PIN_FILE.exists() else PIN_FILE.read_text().strip())
if UPDATE_CODE and existed and not PINNED_COMMIT.strip():
    want = sh(['git', 'rev-parse', 'FETCH_HEAD'], cwd=REPO)
if want:
    sh(['git', 'checkout', '--detach', want], cwd=REPO)
COMMIT = sh(['git', 'rev-parse', 'HEAD'], cwd=REPO)
assert not sh(['git', 'status', '--porcelain'], cwd=REPO), 'El código clonado tiene cambios locales'
if PINNED_COMMIT.strip() or UPDATE_CODE or not PIN_FILE.exists():
    PIN_FILE.parent.mkdir(parents=True, exist_ok=True)
    PIN_FILE.write_text(COMMIT)
print('Código:', GITHUB_URL, '| rama', BRANCH, '| commit', COMMIT)
print('Commit fijado en Drive:', PIN_FILE.read_text().strip())

# --- Entorno ----------------------------------------------------------------------------------------
child_env = dict(os.environ, PYTHONPATH=str(REPO / 'src'),
    AUTO_PROMPT_PERSIST_ROOT=PERSIST_ROOT, PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8', HF_HUB_DISABLE_TELEMETRY='1',
    AUTO_PROMPT_EXECUTION_CONTEXT='google_colab_user_notebook')
try:   # HF_TOKEN se lee de los secretos de Colab (activa "Acceso desde el notebook"); nunca se imprime ni se guarda
    from google.colab import userdata
    child_env['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN leído de los secretos de Colab (valor no mostrado).')
except Exception as error:
    print('HF_TOKEN no disponible (' + type(error).__name__ + '): SAM es público; solo DINOv3 lo necesita.')
LOG_DIR = Path(PERSIST_ROOT) / 'e2/notebook_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

def command(args, log_name):
    """Ejecuta un proceso mostrando su salida en vivo y guardándola en Drive."""
    with (LOG_DIR / log_name).open('w', encoding='utf-8') as log:
        process = subprocess.Popen(args, cwd=REPO, env=child_env, text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        result = process.wait()
    if result:
        raise RuntimeError(f'Proceso terminado con código {result}; revisa la salida y el log {log_name}')

if not PYTHON.is_file():
    command([sys.executable, str(REPO / 'scripts/bootstrap_colab.py'), '--venv', str(PYTHON.parent.parent)], 'bootstrap.log')
command([str(PYTHON), '-m', 'pip', 'install', '--no-deps', '-r', str(REPO / 'requirements-stage2.txt')], 'stage2_pins.log')
command([str(PYTHON), '-m', 'pip', 'check'], 'pip_check.log')
print('Entorno listo.')

In [ ]:
# Comprueba que Colab te dio una GPU (Entorno de ejecución > Cambiar tipo > T4 GPU).
command(['nvidia-smi'], 'nvidia_smi.log')
command([str(PYTHON), '-c', 'import torch; assert torch.cuda.is_available(), "NO HAY GPU: cambia el tipo de entorno a GPU"; print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)'], 'gpu_check.log')

## 2. Datos (etapa 2): descarga y preparación de Sen1Floods11
Descarga los 446 pares a tu Drive, audita, crea splits y fija estadísticas con **solo train**. No abre test ni Bolivia para ajustar nada.

In [ ]:
# Descarga Sen1Floods11 v1.1 (446 pares S2Hand/LabelHand) a TU Drive, audita, crea los splits y fija el preprocesamiento.
# Es la parte más lenta. Es reanudable: si Colab se corta, vuelve a ejecutar las celdas de entorno y esta.
command([str(PYTHON), str(REPO / 'scripts/prepare_stage2.py'), '--root', PERSIST_ROOT,
         '--evidence', str(Path(PERSIST_ROOT) / 'stage2/evidence')], 'stage2_prepare.log')

In [ ]:
# Segunda pasada sin red: relee todo desde tu Drive y comprueba que coincide.
command([str(PYTHON), str(REPO / 'scripts/prepare_stage2.py'), '--root', PERSIST_ROOT, '--offline',
         '--evidence', str(Path(PERSIST_ROOT) / 'stage2/evidence')], 'stage2_offline.log')
report = json.loads((Path(PERSIST_ROOT) / 'stage2/evidence/validation.json').read_text())
print(json.dumps({k: {'counts': v['counts'], 'parameters_reused': v['parameters_reused'], 'reload': v['reload']}
                  for k, v in report['regimes'].items()}, indent=2))
assert report['models_executed'] is False and report['reserved_pixel_caches'] == []

## 3. Línea base (etapa 3)
SAM 2.1 tiny congelado sobre train/val con puntos simulados (1 positivo + 1 negativo por componente de agua). Máscara elegida por IoU **predicho**. Las métricas de esta celda son tuyas y **solo se interpretan tras revisar tus salidas**.

In [ ]:
# Reutiliza scripts/run_stage3.py: SAM 2.1 tiny congelado, puntos simulados desde etiquetas (referencia asistida), evaluador común.
MODE3 = 'prepare'
command([str(PYTHON), str(REPO / 'scripts/run_stage3.py'), MODE3, '--root', PERSIST_ROOT], 'stage3_' + MODE3 + '.log')

In [ ]:
RUNS3 = sorted(p for p in (Path(PERSIST_ROOT) / 'stage3/evidence').iterdir() if p.is_dir() and (p / 'validation.json').is_file())
assert RUNS3, 'No hay run completo de la etapa 3; revisa e2/notebook_logs/stage3_prepare.log'
R3 = json.loads((RUNS3[-1] / 'validation.json').read_text())
print('run_id:', R3['run_id'], '| estado:', R3['status'])
for regime, value in R3['regimes'].items():
    v = value['summary']['val']['pooled']['values']
    print(regime, '-> val agrupado: IoU agua', v['water_iou'], '| precisión', v['precision'], '| recall', v['recall'], '| Dice', v['dice'])

## 4. Teardown del modelo base (etapa 4, solo SAM)
Parámetros y FLOPs por bloque, features intermedias, mapa de campo receptivo, latencia y memoria máxima en la GPU que te asignó Colab.

In [ ]:
# Teardown del modelo base (SAM 2.1 tiny): parámetros/FLOPs por bloque, features intermedias, ERF, latencia y memoria.
# Solo SAM: DINOv3 es opcional y no bloquea E2.
command([str(PYTHON), str(REPO / 'scripts/run_stage4.py'), 'prepare', '--models', 'sam', '--root', PERSIST_ROOT], 'stage4_sam_prepare.log')

In [ ]:
from IPython.display import display, Image, HTML
import csv, html
RUNS4 = sorted(p for p in (Path(PERSIST_ROOT) / 'stage4/evidence').iterdir() if p.is_dir() and (p / 'validation.json').is_file())
assert RUNS4, 'No hay run completo de la etapa 4; revisa e2/notebook_logs/stage4_sam_prepare.log'
RUN4 = RUNS4[-1]
R4 = json.loads((RUN4 / 'validation.json').read_text())
print('run_id:', R4['run_id'], '| modelos:', R4['identity'].get('models_run'))
print(json.dumps(R4['interpretation']['observed'], indent=1)[:3000])
for table in sorted(RUN4.glob('sam_*_parameters.csv'))[:1] + sorted(RUN4.glob('sam_*_flops.csv'))[:1] + sorted(RUN4.glob('sam_*_timings.csv'))[:1]:
    rows = list(csv.reader(table.open(encoding='utf-8', newline='')))
    print(table.name)
    display(HTML('<table>' + ''.join('<tr>' + ''.join('<td>' + html.escape(v) + '</td>' for v in row) + '</tr>' for row in rows) + '</table>'))
for figure in sorted(RUN4.glob('sam_*_0_figure.png'))[:1]:
    print(figure.name)
    display(Image(filename=str(figure)))
print('Evidencia completa:', RUN4.with_suffix('.zip'))

### (Opcional) Incluir DINOv3
No es necesario para E2 y no bloquea nada. Requiere acceso aprobado a DINOv3 en Hugging Face y el secreto `HF_TOKEN`.

In [ ]:
RUN_DINO = False   # True = repetir el teardown incluyendo DINOv3 (gated; requiere HF_TOKEN con acceso aprobado). NO es necesario para E2.
if RUN_DINO:
    command([str(PYTHON), str(REPO / 'scripts/run_stage4.py'), 'prepare', '--models', 'sam,dino', '--root', PERSIST_ROOT], 'stage4_sam_dino_prepare.log')
else:
    print('DINOv3 omitido (opcional).')

## Siguiente
Abre `E2_02_intervencion_decoder.ipynb` (misma carpeta `notebooks/` de la rama `_E.V_`). Guarda **este cuaderno ejecutado** (Archivo → Descargar → .ipynb) y no cambies de commit entre notebooks (queda fijado en `e2/code_commit.txt`). Nada de esto marca E2 como acreditada.